# **Written Report**

## **Heuristic Description**

Our evaluation function estimates how favourable a board position is by combining several chess strategies into a single score. A positive score means White has an advantage, a negative score means Black has an advantage, while zero means the position is roughly equal. The function is built from five components, each representing a seperate aspect of chess strategy.

### **MATERIAL COUNT**

Each piece is assigned a centipawn value reflecting its particular strength. A queen is given a value of 900, which is worth roughly nine pawns (100), a rook is worth 500, bishops and knights are worth 300. The bot aims to avoid losing pieces, since losing material reduces its evaluation score. Possessing more material usually gives a strong advantage.

In [ ]:
for piece_type, value in PIECE_VALUES.items():
    whiteMaterial += len(board.pieces(piece_type, chess.WHITE)) * value
    blackMaterial += len(board.pieces(piece_type, chess.BLACK)) * value

whiteScore += whiteMaterial
blackScore += blackMaterial
score = whiteScore - blackScore

### **PIECE-SQUARE TABLES (PST)** 

A piece on a good square is worth more than the same piece on a bad square. PSTs represent this by adding a positional bonus depending on where each piece is on the board. A knight in the centre receives +20 while a knight on the rim gets a penalty of -50, which is logical considering a knight on the rim gives a disadvantage. We use chess.square_mirror() for Black so each side mirrors the table from their own perspective. The king is assigned two seperate tables, one for early stages of the game where it shouldn't be visible to the opposition, and one for the endgame where it should centralise and become more active.

In [ ]:
def get_pst_index(table, color, square) -> int:
    if color == chess.WHITE:
        return table[square]
    else:
        return table[chess.square_mirror(square)]

def evaluatePSTs(board, color, endgamePhaseWeight):
    value = 0
    value += evaluatePST(PAWN_TABLE,   board.pieces(chess.PAWN,   color), color)
    value += evaluatePST(KNIGHT_TABLE, board.pieces(chess.KNIGHT, color), color)
    value += evaluatePST(BISHOP_TABLE, board.peices(chess.BISHOP, color), color)
    value += evaluatePST(ROOK_TABLE,   board.pieces(chess.ROOK,   color), color)
    value += evaluatePST(QUEEN_TABLE,  board.pieces(chess.QUEEN,  color), color)

    kingMid = get_pst_index(KING_TABLE_MID, color, board.king(color))
    kingEnd = get_pst_index(KING_TABLE_END, color, board.king(color))
    
    value += kingMid * (1 - endgamePhaseWeight) + kingEnd * endgamePhaseWeight
    return value
    

### **MOBILITY**

We added mobility to reward positions where a side has more legal moves. This encourages active pieces and helps stop the bot from playing too passively or repeating the same safe moves. In chess, having more mobility usually means your pieces are better placed and you have more options.

In [ ]:
whiteScore += 2 * mobility_score(board, chess.WHITE)
blackScore += 2 * mobility_score(board, chess.BLACK)

### **GAME PHASE DETECTION (Endgame Weight)**

Strategies in chess can change significantly between the middlegame and endgame so we calculate an endgame phase weight between 0.0 (full middlegame) and 1.0 (full endgame) based on how much non-pawn material remains on the board. As a king is vulnerable in the middlegame and powerful in the endgame, this weight is used to blend the king's middlegame and endgame PST scores and decrease king safety penalties as it gets nearer to the endgame. Pawns are not involved in this calculation as their presence is not significant to whether king is safe.

In [ ]:
endgame_material_start = PIECE_VALUES[chess.ROOK] * 2 + PIECE_VALUES[chess.BISHOP] + PIECE_VALUES[chess.KNIGHT]

def endgame_phase_weight(material_without_pawns) -> float:
    multiplier = 1 / endgame_material_start
    return 1 - min(1, material_without_pawns * multiplier)

### **KING SAFETY**

A king is vulnerable in the middlegame, and can be checkmated quickly. So we reward castling (+30) because it moves the king behind its own pawns and to a safer corner. Each pawn remaining in front of the castled king adds +5 as these form a shield against attacks. A penalty of -7 is given if the bot loses its castling right (-7 per side) since it is no longer looking to get safe. A king in the centre is penalised -30 as it is vulnerable to open file attacks from rooks and queens. The entire king safety score is multiplied by (1 - endgamePhaseWeight), fading to zero as endgame approaches as king centralisation is important.

In [ ]:
def KingSafety(board, color, endgamePhaseWeight):
    score = 0
    king_sq = board.king(color)

    if board.has_kingside_castling_rights(color): score += 5
    else:  score -= 7
    if board.has_queenside_castling_rights(color): score += 5
    else:  score -= 7

    if king_sq in KING_CASTLE_POSITION:
        score += 30
        rook_sq = ROOK_CASTLE_POSITION[king_sq]
        rook_piece = board.piece_at(rook_sq)
        if rook_piece and rook_piece.piece_type == chess.ROOK and rook_piece.color == color:
            score += 15
        for pawn_sq in PAWN_CASTLE_POSITION[king_sq]:
            if board.piece_type_at(pawn_sq) == chess.PAWN:
                score += 5

    if king_sq in KING_CENTER_POSITION[color]:
        score -= 30

    return score * (1 - endgamePhaseWeight)

### **MOP-UP SCORE**

A bot having more material in the endgame is not enough on its own to win, so this function converts the advantage into checkmate. The mop-up score will activate only when ahead by two or more pawns worth of material and the endgame phase weight is non-zero. Pushing the enemy king to the edge of the board as they have fewer ways to escape and bringing the friendly king closer to the enemy king to assist with completing a checkmate is rewarded. A winning bot might shuffle pieces indefinitely without making progress without this component, leading to a draw by the fifty-move rule.

In [ ]:
def mopUpScore(friendlyKingSq, enemyKingsq, myMaterial, enemyMaterial, endgamePhaseWeight) -> int:
    eval = 0
    if myMaterial > enemyMaterial + PIECE_VALUES[chess.PAWN] * 2 and endgamePhaseWeight > 0:
        eval += CENTER_MANHATTAN_DISTANCE[enemyKingSq] * 10
        eval += (14 - chess.square_manhattan_distance(friendlyKingSq, enemyKingSq)) * 4
    
    return eval * endgamePhaseWeight

**Why We Chose These Components**

Material gives the bot a strong base for judging who is ahead.

Piece-square tables help it understand positional play.

Mobility encourages active and flexible positions.

King safety helps it avoid dangerous king placement.

Mop-up scoring improves its behaviour in simplifid winning endgames.

**Simplified Version of evaluate() Code**

In [ ]:
whiteScore += material
whiteScore += mobility
whiteScore += piece_square_tables
whiteScore += king_safety
whiteScore += mop_up

blackScore += material
blackScore += mobility
blackScore += piece_square_tables
blackScore += king_safety
blackScore += mop_up

score = whiteScore - blackScore

## **Depth Analysis**

We looked at how the depth of the minimax search affected the bot. This was important as the strength of a heuristic did not stay the same at every depth.

At depth 2, the bot could only look a small number of moves ahead, so it relied more on the immediate score given by the evaluate function. In these faster tests, simpler evaluation ideas often looked more stable as the bot was mainly reacting to short-term positional features as opposed to long-term plans.

At depth 4, the bot could search further ahead, so longer-term ideas became more useful. This was especially evident in simplified endgames. In those positions, the mop-up term was more helpful because the bot had enough search depth to make use of the extra endgame guidance. This meant that the value of the heuristic depended partly on the search depth being used.

Our testing showed that minimax depth affected both the outcome of games and how useful certain evaluation terms were. A simpler heuristic could perform well at shallow depth, but the mop-up heuristic became more useful when the bot searched deeper and position was simplified. 

## **Heuristic Comparison**

To improve our chess bot, we designed and tested more than one evaluation heuristic rather than relying on a single version from the start. Our aim was to compare whether adding more chess knowledge to the evaluation function would actually improve practical play.

### **Heuristic 1:**

 The first heuristic used the components material balance, piece-square tables, mobility and king safety. Material balance tells the bot who is ahead in pieces, piece-square tables reward good placement of pieces, mobility rewards positions where a side has more legal moves, and king safety rewards castled or protected kings.

### **Heuristic 2:**

The second heuristic kept all parts of Heuristic 1, but added an extra mop-up term for endgames.
The purpose of the mop-up term was to help the bot convert winning endgames more effectively. When one side is materially ahead and the game has simplified, this heuristic rewards pushing the enemy king towards the edge of the board and bringing the friendly king closer to the enemy king.
The idea is that a normal positional evaluation can recognise that a side is winning, but it may not always guide the bot towards the fasted or cleanest way to finish the game. The mop-up term was added to give the bot more specific endgame knowledge.

We expected Heuristic 2 to perform better because it included all of the features from Heuristic 1, while also adding an extra mop-up term for simplified endgames. This was intended to help the bot convert winning positions more efficiently by guiding the king towards a more active endgame role and restricting the opposing king.

### **Testing Method**

We tested the bots from several positions, such as the normal starting position, common opening setups such as the Italian Game, Queen's Gambit Declined, Caro-Kann, King's Indian structure, and Scotch setup and simplified endgames such as KQ vs K and KR vs K with each version played as both White and Black.
In broader depth-2 benchmark testing across several positions, the simpler heuristic without mop-up had a slight edge overall. However, the mop-up version performed well in depth-4 endgame testing, indicating that the added endgame term is more beneficial when the bot can search further ahead.

### **Examples**

A clear pattern from testing was that the version with mop-up handled simplified winning positions more effectively. In endgame situations like KQ vs K and KR vs K, the mop-up version showed better behaviour in trying to restrict the opposing king and activate its own king, exactly the goal of the heuristic.
Also, from the move logs we could see that the simpler heuristic could sometimes play more passively in winning positions, while the mop-up version was more likely to move towards an actual conversion plan in the endgame.

**Without mop-up at depth 2: 8.5 / 16 = 53.1%**

**With mop-up at depth 2: 7.5 / 16 = 46.9%**

**With mop-up in endgame-only tests: 2.5 / 4 = 62.5%**

Overall we determined that Heuristic 2, the version with mop-up, was a better choice. Although the simpler heuristic had a small advantage in the broader depth-2 benchmark, we selected the mop-up version as our final bot because it performed well in deeper endgame focused testing, which is the phase of the game that heuristic was specifically designed to improve.